# Capítulo 8 — Transferência de Calor em Solos

**Autor:** Jader Lugon Junior  
**Livro:** Fenômenos de Transporte: Fundamentos e Modelagem Computacional  
**Repositório:** [github.com/JaderLugon/fenomenos-transporte-livro](https://github.com/JaderLugon/fenomenos-transporte-livro)

---

## 🎯 Objetivos de Aprendizagem

Ao final deste notebook, você será capaz de:

1. **Estimar** a condutividade térmica efetiva $k(\theta)$ a partir da textura e conteúdo de água
2. **Aplicar** os modelos de Côté & Konrad, Johansen e P-EMA
3. **Calcular** a difusividade térmica $\alpha = k/(\rho_b c_p)$
4. **Determinar** a profundidade de penetração térmica para ciclos diurnos e anuais
5. **Implementar** esquema implícito por Diferenças Finitas para a equação de calor
6. **Interpretar** o balanço energético na interface solo-atmosfera

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('\u2705 Ambiente configurado com sucesso!')

---

## 🌡️ Seção 1: Condutividade Térmica Efetiva

### Exercício 1.1: Modelo de Côté & Konrad

Calcule $k(\theta)$ para um solo franco-arenoso com $\theta = 0,20$, usando:

- $\kappa = 1,5$, $k_{\text{seco}} = 0,3$ W/(m·K), $k_{\text{sat}} = 1,8$ W/(m·K)
- $\theta_r = 0,058$, $\theta_s = 0,41$

**Modelo:**

$$S_e = \frac{\theta - \theta_r}{\theta_s - \theta_r}, \quad K_e = \frac{\kappa \cdot S_e}{1 + (\kappa - 1) \cdot S_e}$$

$$k(\theta) = k_{\text{sat}}^{K_e} \cdot k_{\text{seco}}^{1 - K_e}$$

In [ ]:
theta = 0.20
theta_r, theta_s = 0.058, 0.41
k_seco, k_sat = 0.3, 1.8
kappa = 1.5

S_e = (theta - theta_r) / (theta_s - theta_r)
K_e = (kappa * S_e) / (1 + (kappa - 1) * S_e)
k_eff = k_sat**K_e * k_seco**(1 - K_e)

print('Se = {:.4f}'.format(S_e))
print('Ke = {:.4f}'.format(K_e))
print('k(theta) = {:.3f} W/(m*K)'.format(k_eff))
print('\nA agua nos poros atua como ponte termica, aumentando k de {:.1f} para {:.2f}'.format(k_seco, k_eff))

---

### Exercício 1.2: Comparação de Modelos

In [ ]:
def k_cote_konrad(theta, theta_r, theta_s, k_seco, k_sat, kappa):
    S_e = np.clip((theta - theta_r) / (theta_s - theta_r), 0, 1)
    K_e = (kappa * S_e) / (1 + (kappa - 1) * S_e)
    return k_sat**K_e * k_seco**(1 - K_e)

def k_johansen(theta, theta_s, k_seco, k_sat):
    return k_seco + (k_sat - k_seco) * np.clip(theta / theta_s, 0, 1)

def k_pema(theta, theta_s, k_seco, k_sat, p_c=0.3):
    S_r = np.clip(theta / theta_s, 0, 1)
    return np.where(S_r < p_c, k_seco, k_seco + (k_sat - k_seco) * ((S_r - p_c)/(1 - p_c))**2)

theta_range = np.linspace(theta_r, theta_s, 50)

fig, ax = plt.subplots()
ax.plot(theta_range, k_cote_konrad(theta_range, theta_r, theta_s, k_seco, k_sat, 1.5), 'b-', lw=2, label='Cote & Konrad')
ax.plot(theta_range, k_johansen(theta_range, theta_s, k_seco, k_sat), 'r--', lw=2, label='Johansen')
ax.plot(theta_range, k_pema(theta_range, theta_s, k_seco, k_sat), 'g-.', lw=2, label='P-EMA')
ax.set_xlabel('Conteudo de Agua, theta [m3/m3]')
ax.set_ylabel('Condutividade Termica, k [W/(m*K)]')
ax.set_title('Modelos de Condutividade Termica Efetiva', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 🌊 Seção 2: Difusividade Térmica e Profundidade de Penetração

### Exercício 2.1

Para um solo com $k = 1,2$ W/(m·K), $\rho_b = 1600$ kg/m³, $c_p = 900$ J/(kg·K), calcule:

$$\alpha = \frac{k}{\rho_b \cdot c_p}, \quad d = \sqrt{\frac{\alpha \cdot P}{\pi}}$$

In [ ]:
k = 1.2
rho_b = 1600
c_p = 900

alpha = k / (rho_b * c_p)
d_diurno = np.sqrt(alpha * 86400 / np.pi)
d_anual = np.sqrt(alpha * 365.25 * 86400 / np.pi)

print('alpha = {:.2e} m2/s'.format(alpha))
print('Penetracao diurna: d = {:.1f} cm'.format(d_diurno * 100))
print('Penetracao anual:  d = {:.1f} m'.format(d_anual))

---

## 📊 Seção 3: Perfil Térmico Diurno

### Solução Analítica

$$T(z, t) = T_{\text{média}} + A \cdot \exp\left(-z\sqrt{\frac{\omega}{2\alpha}}\right) \cdot \sin\left(\omega t - z\sqrt{\frac{\omega}{2\alpha}}\right)$$

In [ ]:
T_media = 25.0
A_amp = 10.0
omega = 2 * np.pi / 86400
alpha_sol = 8.33e-7
beta = np.sqrt(omega / (2 * alpha_sol))
z = np.linspace(0, 0.5, 100)

fig, ax = plt.subplots()
for t_h in [0, 4, 8, 12, 16, 20]:
    T_z = T_media + A_amp * np.exp(-beta * z) * np.sin(omega * t_h * 3600 - beta * z)
    ax.plot(T_z, z * 100, lw=2, label='t = {}h'.format(t_h))

ax.set_xlabel('Temperatura [C]')
ax.set_ylabel('Profundidade [cm]')
ax.set_title('Perfil Termico Diurno no Solo', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.invert_yaxis()
plt.tight_layout()
plt.show()
print('Profundidade de penetracao: {:.1f} cm'.format(1/beta * 100))

---

## 🧮 Seção 4: Esquema Implícito por Diferenças Finitas

### Exercício 4.1: Montagem da Matriz

$$-r \cdot T_{i-1}^{n+1} + (1 + 2r) \cdot T_i^{n+1} - r \cdot T_{i+1}^{n+1} = T_i^n$$

onde $r = \alpha \Delta t / \Delta z^2$.

In [ ]:
alpha_fd = 8.33e-7
dz = 0.01
dt = 300
r = alpha_fd * dt / dz**2

print('Delta z = {:.2f} cm'.format(dz * 100))
print('Delta t = {} s'.format(dt))
print('r (Fourier) = {:.4f}'.format(r))

a = -r
b = 1 + 2*r
c = -r
print('\nCoeficientes:')
print('  a (sub)     = {:.4f}'.format(a))
print('  b (diagonal) = {:.4f}'.format(b))
print('  c (super)   = {:.4f}'.format(c))
print('  Dominancia: |b| - |a| - |c| = {:.4f} > 0 -> OK'.format(abs(b) - abs(a) - abs(c)))

---

## 🔬 Estudos de Caso

| Estudo de Caso | Descrição | Link |
|----------------|-----------|------|
| **Caso 8.1** | Perfil Térmico Diurno em Solo | [🔗 Abrir](../casos/08_1_Perfil_Termico_Diurno_Solo.ipynb) |
| **Caso 8.2** | Estabilidade Numérica | [🔗 Abrir](../casos/08_2_Estabilidade_Numerica_Diferencas_Finitas.ipynb) |

---

## 📖 Referências

- CÔTÉ, J.; KONRAD, J.-M. (2005). *Canadian Geotechnical Journal*, 42(2), 472-483.
- JOHANSEN, O. (1975). *Thermal Conductivity of Soils*. PhD thesis.
- DE VRIES, D. A. (1963). *Physics of Plant Environment*.

---

## 🔄 Navegação

- [📚 Capítulo Anterior](07_Fundamentos_Calor.ipynb)
- [📚 Próximo Capítulo](09_Trocadores_Calor.ipynb)

In [ ]:
print('Capitulo 8 concluido com sucesso!')